# Filtrado colaborativo usuario - usuario basado en K-NN

Predice la valoración de un libro para un usuario a partir de los usuarios más similares. Aquí se utiliza KNNWithMeans con similitud coseno para medir qué tan parecidos son los perfiles de valoración.

Se toma un muestra de 10.000 usuarios para reducir el uso de memoria y garantizar que el entrenamiento de K-NN sea computacionalmente viable en este entorno.

En un escenario de producción, convendría aplicar técnicas de reducción de dimensionalidad o uso de índices aproximados para escalar a todo el dataset.

In [2]:
import pandas as pd
import numpy as np

In [4]:
ratings_df = pd.read_csv("ratings_limpios.csv")
resumen_usuario = pd.read_csv("resumen_usuario.csv")

In [5]:
np.random.seed(42)

usuarios_unicos = resumen_usuario['User-ID'].unique()

# 10000 o el dataset completo si llegara a tener menos de 10000
n_muestra = min(10000, len(usuarios_unicos))

usuarios_muestra = np.random.choice(usuarios_unicos, size=n_muestra, replace=False)

sub_df = ratings_df[ratings_df['User-ID'].isin(usuarios_muestra)].copy()


In [58]:
from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))

surprise_df = sub_df.copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

# 0.1, 0.2, 0.3
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 20, 40, 60
# cosine, pearson, msd
algo = KNNWithMeans(k=40, sim_options={'name': 'msd', 'user_based': True}, verbose=False)
algo.fit(trainset)

predictions = algo.test(testset)

In [59]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID':pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [60]:
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)


df_evaluacion = pd.merge(
    metricas_usuarios,
    resumen_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [61]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.462925  19.323344  12.962964  0.978052
Jóvenes       1.465101  17.371094  12.202692  0.981781
Mayores       1.370991  13.883117  10.957033  0.987262

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.400965   7.759901   7.655392  0.999354
1                1.496622  21.126063  13.859330  0.973910

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.573307   8.258427   6.617082  0.985848
1                1.255787  18.156311  12.517988  0.979552
2                1.886988  17.730366  13.462403  0.995092

Obserbaciones:

Grupo Etario:

Ek desempeño es bastante homogéneo. los mayores presentan el menor MAE y los góvenes el mayor, aunque las diferencias son pequñas. El NDCG es alto en los tres grupos.

Grupo separado por historial:

Los usuarios con historial corto tienen meno MAE y un NDCG casi perfecto, pero sus valores bajos de CG/DCG se deben probablemente a que tienen pocos ítems evaluados. Los usuarios con historial largo reciben mayor utilidad acumulada, aunque con un MAE ligeramente superior y menor NDCG (Seguramente cuesta encontrar vecinos que coincidan exactamente en todo su historial).

Grado de exigencia:

Los usuarios exigentes presentan claramente el mayor MAE, por lo que el modelo predice peor sus calificaciones. Los usuarios normales tienen el mejor MAE. El NDCG es alto en todos, aunque los genereosos obtienen el mejor valor.


### Hipótesis: los usuarios con un historial de interacciones largo se ven más beneficiados por el sistema de recomendación que aquellos con un historial corto.

In [10]:
from scipy.stats import shapiro

g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_corto)
    print(f"Test de Shapiro-Wilk — Historial corto — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_largo)
    print(f"Test de Shapiro-Wilk — Historial largo — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — Historial corto — MAE: estadístico=0.832, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — MAE: estadístico=0.922, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — Historial corto — CG@10: estadístico=0.691, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — CG@10: estadístico=0.722, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — Historial corto — DCG@10: estadístico=0.895, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — DCG@10: estadístico=0.835, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — Historial corto — NDCG@10: estadístico=0.078, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — NDCG@10: estadístico=0.598, p-valor=0.000


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [11]:
import scipy.stats as stats

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    stat, p = stats.levene(v_largo,v_corto)
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=26.133, p-valor=0.000
Test de Levene para CG@10: Estadístico=478.911, p-valor=0.000
Test de Levene para DCG@10: Estadístico=516.272, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=312.106, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis.

In [12]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_largo, v_corto)
    print(f"\nTest de Kruskal-Wallis - {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre historiales largos y cortos.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre historiales largos y cortos.")


Test de Kruskal-Wallis - MAE: estadístico=21.061, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre historiales largos y cortos.

Test de Kruskal-Wallis - CG@10: estadístico=632.748, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - DCG@10: estadístico=584.995, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - NDCG@10: estadístico=522.377, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre historiales largos y cortos.


En las cuatro métricas, el p-valor es de 0.000, lo que confirma la existencia de diferencias estadísticamente significativas entre el grupo de historial corto y el de historial largo.

Aunque los usarios con historial corto registran un MAE ligeramente menor y un NDCG cercano a 1, los usuarios con un historial de interacciones largo reciben un beneficio mayor en volumen y posicionamiento de recomendación relevante. Por lo que podemos confirmar la hipótesis planteada.

### Hipótesis: El modelo presenta un rendimiento similar en los distintos grupos etarios

In [23]:
from scipy.stats import shapiro

g_joven = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Jóvenes"]
g_adulto = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Adultos"]
g_mayores = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Mayores"]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_joven)
    print(f"Test de Shapiro-Wilk — jóvenes — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_adulto)
    print(f"Test de Shapiro-Wilk — adultos — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_mayores)
    print(f"Test de Shapiro-Wilk — mayores — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — jóvenes — MAE: estadístico=0.879, p-valor=0.000
Test de Shapiro-Wilk — adultos — MAE: estadístico=0.912, p-valor=0.000
Test de Shapiro-Wilk — mayores — MAE: estadístico=0.882, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — jóvenes — CG@10: estadístico=0.656, p-valor=0.000
Test de Shapiro-Wilk — adultos — CG@10: estadístico=0.672, p-valor=0.000
Test de Shapiro-Wilk — mayores — CG@10: estadístico=0.648, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — jóvenes — DCG@10: estadístico=0.788, p-valor=0.000
Test de Shapiro-Wilk — adultos — DCG@10: estadístico=0.790, p-valor=0.000
Test de Shapiro-Wilk — mayores — DCG@10: estadístico=0.792, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — jóvenes — NDCG@10: estadístico=0.497, p-valor=0.000
Test de Shapiro-Wilk — adultos — NDCG@10: estadístico=0.544, p-valor=0.000
Test de Shapiro-Wilk — mayores — NDCG@10: estadístico=0.457, p-valor=0.000


MAE y NDCG: Cumplen el supuesto de homocedasticidad. No hay evidencia estadística de que las varianzas entre los grupos sean distintas.

CG y DCG: Violan el supuesto de homocedasticidad. Las varianzas de los grupos son significativamente diferentes enrte sí.

Aplicamos:
* ANOVA para MAE y NDCG
* Kruskal-Wallis para CG y DCG

In [ ]:
from scipy import stats

alpha = 0.05

metricas_anova = ['MAE', f'NDCG@{K}']

for metrica in metricas_anova:
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()

    stat, p = stats.f_oneway(v_joven, v_adulto, v_mayores)
    print(f"\nTest de ANOVA — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")
    
    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos etarios.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos etarios.")


Test de ANOVA — MAE: estadístico=0.273, p-valor=0.761
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en MAE entre los grupos etarios.

Test de ANOVA — NDCG@10: estadístico=2.419, p-valor=0.089
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en NDCG@10 entre los grupos etarios.


In [28]:
metricas_kw = [f'CG@{K}', f'DCG@{K}']

for metrica in metricas_kw:
    # Extracción de valores limpios
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_joven, v_adulto, v_mayores)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos etarios.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos etarios.")


Test de Kruskal-Wallis — CG@10: estadístico=2.768, p-valor=0.251
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en CG@10 entre los grupos etarios.

Test de Kruskal-Wallis — DCG@10: estadístico=1.775, p-valor=0.412
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en DCG@10 entre los grupos etarios.


Los resultados de los tests Kruskal-Wallis y ANOVA confirman que no existen diferencias estadísticamente significativas entre los grupos para todas las métricas.

El sistema de recomendación demuestra un comportamiento equitativo y sesgo insignificate respecto a la edad del usuario.

### Hipótesis: El modelo presenta un rendimiento diferente según el grado de exigencia del usuario.

In [13]:
from scipy.stats import shapiro

#grupo de exigencia: 0 Exigente, 1 normal, 2 generoso
g_exigente = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 0]
g_normal = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 1]
g_generoso = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 2]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_exigente)
    print(f"Test de Shapiro-Wilk — exigente — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_normal)
    print(f"Test de Shapiro-Wilk — normal — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_generoso)
    print(f"Test de Shapiro-Wilk — generoso — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — exigente — MAE: estadístico=0.921, p-valor=0.000
Test de Shapiro-Wilk — normal — MAE: estadístico=0.899, p-valor=0.000
Test de Shapiro-Wilk — generoso — MAE: estadístico=0.746, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — exigente — CG@10: estadístico=0.578, p-valor=0.000
Test de Shapiro-Wilk — normal — CG@10: estadístico=0.645, p-valor=0.000
Test de Shapiro-Wilk — generoso — CG@10: estadístico=0.484, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — exigente — DCG@10: estadístico=0.768, p-valor=0.000
Test de Shapiro-Wilk — normal — DCG@10: estadístico=0.755, p-valor=0.000
Test de Shapiro-Wilk — generoso — DCG@10: estadístico=0.565, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — exigente — NDCG@10: estadístico=0.357, p-valor=0.000
Test de Shapiro-Wilk — normal — NDCG@10: estadístico=0.531, p-valor=0.000
Test de Shapiro-Wilk — generoso — NDCG@10: estadístico=0.353, p-valor=0.000


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [14]:
import scipy.stats as stats

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    stat, p = stats.levene(v_exigente, v_normal, v_generoso, center='median')
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=42.675, p-valor=0.000
Test de Levene para CG@10: Estadístico=30.477, p-valor=0.000
Test de Levene para DCG@10: Estadístico=35.153, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=24.568, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis

In [15]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_exigente, v_normal, v_generoso)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos de exigencia.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos de exigencia.")


Test de Kruskal-Wallis — MAE: estadístico=603.032, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre los grupos de exigencia.

Test de Kruskal-Wallis — CG@10: estadístico=466.385, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — DCG@10: estadístico=558.236, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — NDCG@10: estadístico=78.431, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre los grupos de exigencia.


En las cuatro métricas evaluadas, el p-valor es de 0.000. Esto permite rechazar la hipótesis nula y confirmar la existencia de diferencias significativas según la exigencia del usuario.

El sistema presenta una penalización muy clara ante usuarios altamente exigentes, donde el error de predicción prácticamente se duplica y la cantidad de contenido relevante cae abruptamente.

# Conclusión general de kNN usuario - usuario

Obtiene valores altes de NDCG, por lo que ordena adecuadamente las recomendaciones dentro del conjunto evaluado. El desempeño por edad es relativamente homogéneo. No se observaron diferencias significativas en MAE entre usuarios de historial corto y largo, aunque sí existen diferencias en las métricas de ganancia acumulada. Su principal dificultad aparece en los usuarios exigentes, para quienes presenta un MAE considerablemente mayor.



# Impacto en modificaciones simples:

## Valores base
* test = 0.2
* KNN = 40
* Similitud = cosine

#### Grupo Etario  
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.460 | 19.320 | 12.961 | 0.978 |
| Jóvenes | 1.466 | 17.365 | 12.194 | 0.981 |
| Mayores | 1.375 | 13.883 | 10.957 | 0.987 |

#### Historial de Interacciones 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto|1.400|7.759|7.663|0.999|
|Largo|1.494|21.121|13.855|0.973|

#### Grupo de Exigencia 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.572 | 8.258 | 6.617 | 0.985 |
| Normal | 1.253 | 18.152 | 12.514 | 0.979 |
| Generoso | 1.887 | 17.730 | 13.463 | 0.995 |

## Experimento 1: Cambiar tamaño de train/test

#### Grupo Etario (test = 0.1) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.444 | 17.155 | 12.090 | 0.982 |
| Jóvenes | 1.439 | 14.828 | 11.206 | 0.989 |
| Mayores | 1.224 | 10.130 |  9.133 | 0.996 |

#### Historial de Interacciones (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.408 |  7.652|  7.573 | 0.999 |
|Largo| 1.477 | 17.735| 12.446 | 0.984 |


#### Grupo de Exigencia (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.532 |  7.641 |  6.330 | 0.990 |
| Normal   | 1.286 | 15.952 | 11.602 | 0.984 |
| Generoso | 1.823 | 17.192 | 13.171 | 0.995 |

#### Grupo Etario (test = 0.3) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.471 | 20.642 | 13.480 | 0.975 |
| Jóvenes | 1.459 | 18.964 | 12.895 | 0.980 |
| Mayores | 1.491 | 15.144 | 11.342 | 0.986 |

#### Historial de Interacciones (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.424 |  7.837 |  7.694 | 0.999 |
| Largo | 1.506 | 23.871 | 15.039 | 0.968 |


#### Grupo de Exigencia (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.686 |  8.589 |  6.749 | 0.984 |
| Normal   | 1.226 | 19.484 | 13.069 | 0.976 |
| Generoso | 1.921 | 18.170 | 13.663 | 0.994 |

Efectos detectados:
* A medida que disminuye los datos de entrenamieto, la capacidad para estimar valoraciones exactas empeora ligeramente en la mayoría de las categorías.
* Crecimiento Artificial de la ganancia (CG y DCG): Al aumentar el tamaño del conjunto de prueba, hay una mayor cantidad de ítems relevantes pertencientes a la evaluación.
* El NDCG presenta un leve descenso en casi todos los segmentos a medida que sube el porcentaje de test.
* Las brechas entre subgrupos se mantiene consistente:
    * Los usuarios exigentes  sufren el peor MAE.
    * EL perfil de historial largo tienen mejor CG y DCG a comparación del historial corto.
    * Los adultos y jóvenes retienen un mayor volumen de recomendaciones útiles.

## Experimento 2: Variar K en KNN

#### Grupo Etario (K = 20) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.460 | 19.320 | 12.961 | 0.978 |
| Jóvenes | 1.466 | 17.365 | 12.194 | 0.981 |
| Mayores | 1.375 | 13.883 | 10.957 | 0.987 |

#### Historial de Interacciones (K = 20)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.400 |  7.759 |  7.655 | 0.999 |
|Largo| 1.494 | 21.121 | 13.855 | 0.973 |


#### Grupo de Exigencia (K = 20)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.572 |  8.258 |  6.617 | 0.985 |
| Normal   | 1.253 | 18.152 | 12.514 | 0.979 |
| Generoso | 1.887 | 17.730 | 13.463 | 0.995 |

#### Grupo Etario (K = 60) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.460 | 19.320 | 12.961 | 0.978 |
| Jóvenes | 1.466 | 17.365 | 12.194 | 0.981 |
| Mayores | 1.375 | 13.883 | 10.957 | 0.987 |

#### Historial de Interacciones (K = 60)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.400 |  7.759 |  7.655 | 0.999 |
| Largo | 1.494 | 21.121 | 13.855 | 0.973 |


#### Grupo de Exigencia (K = 60)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.572 |  8.258 |  6.617 | 0.985 |
| Normal   | 1.253 | 18.152 | 12.514 | 0.979 |
| Generoso | 1.887 | 17.730 | 13.463 | 0.995 |

Efectos detectados:
* Los vecinos adicionales no aportan información nueva debido a la dispersión de los datos o a que su similitud es demasiado baja para alterar la predicción.
* Conviene mantener K = 40 o probar valores menores para optimizar recursos computacionales.

## Experimento 3: variar similitud

#### Grupo Etario (pearson) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.444 | 19.303 | 12.936 | 0.977 |
| Jóvenes | 1.439 | 17.396 | 12.208 | 0.981 |
| Mayores | 1.355 | 13.883 | 10.962 | 0.987 |

#### Historial de Interacciones (pearson)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.400 |  7.759 |  7.655 | 0.999 |
|Largo| 1.472 | 21.110 | 13.833 | 0.973 |


#### Grupo de Exigencia (pearson)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.560 |  8.258 |  6.620 | 0.986 |
| Normal   | 1.257 | 18.142 | 12.495 | 0.978 |
| Generoso | 1.880 | 17.727 | 13.457 | 0.994 |

#### Grupo Etario (msd) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.462 | 19.323 | 12.962 | 0.978 |
| Jóvenes | 1.465 | 17.371 | 12.202 | 0.981 |
| Mayores | 1.370 | 13.883 | 10.957 | 0.987 |

#### Historial de Interacciones (msd)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.400 |  7.759 |  7.655 | 0.999 |
| Largo | 1.496 | 21.126 | 13.859 | 0.973 |


#### Grupo de Exigencia (msd)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.573 |  8.258 |  6.617 | 0.985 |
| Normal   | 1.255 | 18.156 | 12.517 | 0.979 |
| Generoso | 1.886 | 17.730 | 13.462 | 0.995 |

Efectos detectados:
* El cambio de la métrica de similitud produce un impacto prácticamente nulo en el ranking de recomendaciones.
* Pearson aporta una leve mejora en la precisión (MAE).
* Las diferencias entre los grupos se mantiene intactas. 
* MSD y Coseno presentan un rendimiento practicamente idénticos.


